# HeritAgent — EXP-0001 kalibrasyon pilotu (Colab, A100 80 GB)

Bu notebook, EXP-0001'den **önce** yapılan kalibrasyon pilotunu çalıştırır (ADR-0003, ADR-0007 E2, ADR-0008).

**Ne yapar:** G0 ajanı, deney havuzlarının hiçbirinde olmayan 30 ALFWorld oyununu sıcaklık 0'da bir kez oynar. Sonra her ailenin ilk oyununu bir kez daha oynar (6 bölüm; sonuçlar birebir aynı mı diye bakılır). Son olarak sahte bir skill bloğuyla tek bir bölüm oynar (kol-B prompt testi). Toplam 37 bölüm.

**Hiçbir şey ayarlamaz.** Yalnızca ölçer: başarı oranı, geçersiz komut ve ayrıştırma hatası oranı, bölüm başına model çağrısı, saatte bölüm sayısı, tekrar belirlenimi. Çıkan sayılar bir EXP-0001 sonucu **değildir**.

**Tahmini süre ve maliyet (M: tahmin, ölçüm değil):**
- kurulum, model indirme ve sunucu açılışı yaklaşık 20–35 dk;
- pilot yaklaşık 10–30 dk;
- toplam yaklaşık 0.5–1 saat. A100'de saatte yaklaşık 7.5 CU ile 4–8 CU eder.

**Nasıl çalıştırılır:**
1. Çalışma zamanı > Çalışma zamanı türünü değiştir > **A100 GPU**. Varsa "Yüksek RAM"i de seçin.
2. Çalışma zamanı > **Tümünü çalıştır**. Drive izni isteyince onaylayın.
3. Bitince son hücre bir özet yazar. Rapor Drive'da `heritagent/pilot/pilot-report.json` olarak durur. **Bu dosyayı bana gönderin**; tavan kararı ve EXP-0001 koşusu ona göre verilir.
4. İş bitince Çalışma zamanı > **Bağlantıyı kes ve çalışma zamanını sil** (CU harcamasını durdurur).

Oturum yarıda koparsa notebook'u yeniden açıp "Tümünü çalıştır" deyin. Pilot Drive'daki kopyasından kaldığı yerden devam eder.

In [ ]:
# Ayarlar
REPO_URL = "https://github.com/sametyilmaztemel/heritagent.git"
CODE_REF = "main"          # pilotun kodu; rapor commit'i kaydeder
DRIVE_ROOT = "/content/drive/MyDrive/heritagent"
WORKERS = 16               # aynı anda oynanan bölüm sayısı (ADR-0008 D3)
MODEL = "Qwen/Qwen2.5-7B-Instruct"
BATCH_INVARIANT = True     # sunucu açılmazsa False yapın ve bana bildirin (ADR-0008 D4)

In [ ]:
# 1) GPU: yalnızca tek bir A100 80 GB (ADR-0008 D1). Başka bir GPU'da hiçbir şey kurulmaz.
import re, subprocess
query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                        "--format=csv,noheader,nounits"], capture_output=True, text=True)
print(query.stdout or query.stderr)
gpus = [line.split(", ") for line in query.stdout.strip().splitlines()]
assert len(gpus) == 1 and "A100" in gpus[0][0] and int(float(gpus[0][1])) >= 76000, (
    "Bu notebook yalnızca TEK bir A100 80 GB ile çalışır. Çalışma zamanı > Çalışma zamanı "
    "türünü değiştir > A100 GPU seçin ve yeniden bağlanın.")
header = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
CUDA_DRIVER = float(re.search(r"CUDA Version:\s*([0-9.]+)", header).group(1))
print("Sürücünün desteklediği en yüksek CUDA:", CUDA_DRIVER)

In [ ]:
# 2) Google Drive: kalıcı kayıtlar burada durur (oturum kapanınca yerel disk silinir).
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

In [ ]:
# 3) Kod: depoyu klonla ve CODE_REF noktasına geç.
import os, subprocess
REPO = "/content/heritagent"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--quiet", "origin"], check=True)
ref = CODE_REF
target = f"origin/{ref}" if subprocess.run(
    ["git", "-C", REPO, "rev-parse", "--verify", "--quiet", f"origin/{ref}"],
    capture_output=True).returncode == 0 else ref
subprocess.run(["git", "-C", REPO, "checkout", "--quiet", "--detach", target], check=True)
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True,
                        text=True, check=True).stdout.strip()
os.chdir(REPO)
print("Kod commit'i:", COMMIT)

In [ ]:
# 4) Kurulum (~5-10 dk): vLLM 0.29.0 + torch 2.13.0, ALFWorld 0.4.2, TextWorld 1.7.0.
# Sürücü CUDA 13'ü destekliyorsa CUDA 13 paketi, desteklemiyorsa resmi CUDA 12.9 paketi.
import subprocess, sys
heritagent = f"{REPO}[alfworld]"
if CUDA_DRIVER >= 13.0:
    command = ["vllm==0.29.0", "-e", heritagent]
else:
    command = ["vllm==0.29.0+cu129", "torch==2.13.0+cu129", "torchvision==0.28.0+cu129",
               "torchaudio==2.11.0+cu129", "-e", heritagent,
               "--extra-index-url", "https://wheels.vllm.ai/0.29.0/cu129",
               "--extra-index-url", "https://download.pytorch.org/whl/cu129"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *command], check=True)
print(subprocess.run([sys.executable, "-c", "import vllm, torch; print('vllm', vllm.__version__, "
                      "'torch', torch.__version__, 'cuda', torch.cuda.is_available())"],
                     capture_output=True, text=True).stdout)

In [ ]:
# 5) ALFWorld verisi (~2-5 dk). Oyunlar, donmuş manifestteki içerik hash'leriyle tek tek
# doğrulanır; tutmayan bir oyun hiç açılmaz.
import os, subprocess
os.environ["ALFWORLD_DATA"] = "/content/alfworld-data"
subprocess.run(["alfworld-download"], check=True, env=os.environ)
DATA_ROOT = "/content/alfworld-data/json_2.1.1"
print(sorted(os.listdir(DATA_ROOT)))

In [ ]:
# 6) Aynı A100 kontrolü, deneyin kendi koduyla (fail-closed).
!python -m evaluation.exp0001.cli gpu-check

In [ ]:
# 7) Model: tek bir revizyon sabitlenir ve dosyaları hash'lenir (~5-10 dk).
# Pilot hub'daki güncel commit'i sabitler ve Drive'a yazar; EXP-0001 koşusu aynısını kullanır.
import json, os
revision_file = f"{DRIVE_ROOT}/model-revision.txt"
revision = open(revision_file).read().strip() if os.path.exists(revision_file) else None
flag = f"--revision {revision}" if revision else ""
!python -m evaluation.exp0001.cli prepare-model --model {MODEL} {flag} --out /content/prepared.json
prepared = json.load(open("/content/prepared.json"))
open(revision_file, "w").write(prepared["revision"])
print("Model revizyonu:", prepared["revision"], "| ağırlık hash'i:", prepared["weights"]["sha256"])

In [ ]:
# 8) Model sunucusu (~5-15 dk): vllm serve, batch-invariant çekirdekler açık (ADR-0008 D4).
# Hazır olunca "the model server ... serves" yazar; günlük: /content/vllm-server.log
# Sunucu açılmazsa günlüğün sonuna bakın: !tail -50 /content/vllm-server.log
bi_flag = "" if BATCH_INVARIANT else "--no-batch-invariant"
!python -m evaluation.exp0001.cli serve --prepared /content/prepared.json --log /content/vllm-server.log --info /content/vllm-server.json {bi_flag}

In [ ]:
# 8b) Ön kontrol (2 model çağrısı, hiçbir deney kaydına girmez): ReAct biçimli bir yanıt ve
# miner'ın JSON şemasıyla yapılandırılmış bir yanıt. Miner normalde ancak 540 lifetime
# bölümünden sonra ilk kez çalışır; burada hata verirse CU harcamadan önce durursunuz.
!python -m evaluation.exp0001.cli preflight --prepared /content/prepared.json

In [ ]:
# 9) Pilot: 30 oyun + 6 tekrar + 1 kol-B testi. Kopyası her dakika Drive'a alınır.
!python -m evaluation.exp0001.cli pilot --prepared /content/prepared.json --data-root {DATA_ROOT} --repo /content/heritagent --server-info /content/vllm-server.json --workers {WORKERS} --out-dir /content/pilot --mirror-dir {DRIVE_ROOT}/pilot/mirror

In [ ]:
# 10) Özet ve EXP-0001 maliyet projeksiyonu (M: pilotun ortalamalarıyla tahmin).
import json, os, shutil
CU_PER_HOUR = 7.5   # Colab'ın A100 için gösterdiği CU/saat değeriyle değiştirin
os.makedirs(f"{DRIVE_ROOT}/pilot", exist_ok=True)
shutil.copy("/content/pilot/pilot-report.json", f"{DRIVE_ROOT}/pilot/pilot-report.json")
report = json.load(open("/content/pilot/pilot-report.json"))
m = report["measures"]
print("G0 başarı oranı (sıcaklık 0):", m["g0_success_rate_p0"])
print("Bölüm başına model çağrısı:", m["model_calls_per_episode"])
print("Saatte bölüm:", m["throughput_episodes_per_hour"])
print("Tekrar belirlenimi (hepsi aynı mı):", m["temperature_0_repeat_determinism"]["all_identical"])
print("Kol-B prompt testi geçti mi:", m["arm_b_rendering_smoke_test"]["passed"])
print()
print(f"{'aday/seed':>9} {'etkileşim':>22} {'bölüm':>7} {'çağrı':>9} {'saat':>7} {'CU':>7}")
for s in report["projection"]["scenarios"]:
    hours = s["active_hours_at_pilot_throughput"]
    print(f"{s['candidates_per_seed']:>9} {s['interaction']:>22} {s['episodes']:>7} "
          f"{s['model_calls_at_pilot_mean'] or '-':>9} "
          f"{'-' if hours is None else round(hours, 1):>7} "
          f"{'-' if hours is None else round(hours * CU_PER_HOUR):>7}")
print("\nRapor Drive'da:", f"{DRIVE_ROOT}/pilot/pilot-report.json", "— bu dosyayı gönderin.")

**Sonra:** `pilot-report.json` dosyasını gönderin. Ben ölçümleri ve projeksiyonu inceleyip hesap tavanı kararını size soracağım (takeover notu §5.12). Ardından `EXP-0001-run.ipynb` ile asıl koşu başlar.

Bitirdiyseniz: Çalışma zamanı > **Bağlantıyı kes ve çalışma zamanını sil**.